<a href="https://colab.research.google.com/github/paulsoriiiano/cmpe-259-project/blob/main/notebooks/parks_virtual_assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# California Parks Virtual Assistant

### Final Project

Fall 2025

CMPE 259 - Natural Language Processing

Author: Paul Junver Soriano

# Description

smth...

# Project Set Up

In [123]:
# Get data and Python modules from repo
!git clone https://github.com/paulsoriiiano/cmpe-259-project
!mv cmpe-259-project/data .
!mv cmpe-259-project/src .
!rm -rf cmpe-259-project

Cloning into 'cmpe-259-project'...
remote: Enumerating objects: 216, done.
remote: Counting objects: 100% (216/216), done.
remote: Compressing objects: 100% (148/148), done.
remote: Total 216 (delta 115), reused 146 (delta 62), pack-reused 0 (from 0)
Receiving objects: 100% (216/216), 600.13 KiB | 2.59 MiB/s, done.
Resolving deltas: 100% (115/115), done.
mv: cannot move 'cmpe-259-project/data' to './data': Directory not empty


In [2]:
# Install required libraries

%%bash
# Vectorstore libraries
pip install faiss-cpu jq

# Huggingface clients
pip install huggingface_hub

# Langchain dependencies
pip install langchain langchain-community langchain_core langchain-huggingface langchain-mistralai langchain-openai

# Weather tool dependencies
!pip install geopy openmeteo_requests requests-cache retry-requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 49.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 757.1/757.1 kB 26.8 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of langchain-community to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of langchain-huggingface to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of langchain-mistralai to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of langchain-openai to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.0/76.0 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 5.0 M

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [22]:
from huggingface_hub import login
from google.colab import userdata

login(new_session=False)
hf_token = userdata.get("HF_TOKEN")         # Get HuggingFace API Token

# Load models

In [124]:
from src.llm_utils import load_chat_model

small_llm = load_chat_model("small")
large_llm = load_chat_model("large")

In [125]:
# Sample query
query = "What is the weather like this weekend in Antelope Valley?"

In [126]:
print(small_llm.invoke(query).content)

 I cannot provide an exact answer without checking the current weather forecast for Antelope Valley. Generally, Antelope Valley in California experiences hot and dry weather during the summer months. However, I would recommend checking a reliable weather source for the most up-to-date and accurate information. You can visit the National Weather Service website or download a weather app to check the forecast for Antelope Valley specifically.


In [127]:
print(large_llm.invoke(query).content)

I'm a large language model, I don't have real-time access to current weather conditions. But I can suggest some ways for you to find out the weather forecast for Antelope Valley this weekend.

You can check online weather websites such as:

1. National Weather Service (NWS): [www.weather.gov](http://www.weather.gov)
2. AccuWeather: [www.accuweather.com](http://www.accuweather.com)
3. Weather.com: [www.weather.com](http://www.weather.com)

You can also check mobile apps like Dark Sky or Weather Underground for hyperlocal weather forecasts.

Additionally, you can search for "Antelope Valley weather forecast" or " Lancaster, CA weather forecast" (if you're referring to the Antelope Valley in California) to get the latest weather updates.

Please note that weather forecasts are subject to change, so it's always a good idea to check the forecast again closer to the weekend for the most up-to-date information.


# Get vector database

In [102]:
from src.vector_db import build_vector_db, load_vector_db

try:
  retriever = load_vector_db().as_retriever()
except:
  retriever = build_vector_db().as_retriever()

## Build RAG chains

In [128]:
from src.rag_pipeline import build_rag_chain

small_rag_chain = build_rag_chain(small_llm, retriever)
large_rag_chain = build_rag_chain(large_llm, retriever)

In [104]:
# Test RAG chain.
query = "Which state beaches allow dogs?"

print(f"Small LLM response: \n\n{small_rag_chain.invoke(query)}")
print("\n======================================================\n")
print(f"Large LLM response: \n\n{large_rag_chain.invoke(query)}")

Small LLM response: 

 Monterey State Beach (South of the Monterey Beach Resort hotel), Asilomar State Beach, Carmel River State Beach, and Garrapata State Park allow dogs on leash. Other state beaches, such as Salinas River State Beach, do not allow dogs at all. Sonoma Coast State Park also has some restrictions on dogs, with certain areas requiring them to be on leash and others prohibiting them entirely due to snowy plover nesting areas. (Sources: Salinas River State Beach information from <https://www.parks.ca.gov/?page_id=573>, Moss Landing State Beach information from <https://www.parks.ca.gov/?page_id=574>, Sonoma Coast State Park information from <https://www.parks.ca.gov/?page_id=451>)


Large LLM response: 

Dogs on leash are allowed at Monterey State Beach (South of the Monterey Beach Resort hotel), Asilomar State Beach, Carmel River State Beach, and Garrapata State Park. 

¹ https://www.parks.ca.gov/?page_id=573 
¹ https://www.parks.ca.gov/?page_id=451


## Build weather function

In [105]:
import src.weather_fn
from src.weather_fn import get_weather

# Test weather function
print(get_weather("Calaveras Big Trees State Park", days=2))

2-day forecast for Calaveras Big Trees State Park:

Day 1: moderate snow
- Temp: 36.8 —— 44.1°F
- Precipitation: 1.39 inches
- Max Wind: 13.8 mph

Day 2: moderate snow
- Temp: 33.8 —— 40.8°F
- Precipitation: 0.17 inches
- Max Wind: 9.0 mph


## Build tools

In [112]:
import re
from langchain.tools import Tool

# RAG tool for different agents
small_rag_tool = Tool(
    name="ParksRAG_SmallLLM",
    func=lambda q: small_rag_chain.invoke(q),
    description="Parks info via small LLM (Mistral-7B)"
)

large_rag_tool = Tool(
    name="ParksRAG_LargeLLM",
    func=lambda q: large_rag_chain.invoke(q),
    description="Parks info via large LLM (Llama-3.3-70B)"
)

# Weather tool
def weather_wrapper(query: str):
  match = re.search(r"(\d+)[ -]?day", query)
  days = int(match.group(1)) if match else 0
  return get_weather(query, 0)

weather_tool = Tool(
    name="WeatherTool",
    func=weather_wrapper,
    description="Get weather info for California parks via OpenMeteo"
)

## Build agents

In [135]:
from langchain.agents import initialize_agent, AgentType

# Group tools
small_tools = [small_rag_tool, weather_tool]
large_tools = [large_rag_tool, weather_tool]

# Initilize agents
small_agent = initialize_agent(
    small_tools,
    llm=small_llm,
    agent_type=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    handle_parsing_errors=True
)

large_agent = initialize_agent(
    large_tools,
    llm=large_llm,
    agent_type=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    handle_parsing_errors=True
)

## Build user queries

In [136]:
small_rag_chain.invoke("Calaveras Big Trees")

' Calaveras Big Trees State Park undergoes seasonal closures. It is open for day use and camping from May 15th until Winter, though the specific dates may vary each year. The park features giant sequoias, pines, flowing rivers and creeks, wildlife, meadows, and offers camping, fishing, hiking, swimming, and more. Dogs are allowed only in the campgrounds and on fire roads. For more information or reservations, visitors can contact (209) 795-2334. The Visitor Center offers a museum, film, and gift shop. The South Grove Trail reopens every year around May 1st. The park permanently protects more than 150,000 acres in California State Parks redwood parks. For additional information, visit SaveTheRedwoods.org or contact the park through its Contact Us page.\n\nSources:\nCalaveras Big Trees State Park information from https://www.parks.ca.gov/?page_id=551\nCalifornia State Parks and CAL FIRE Plan Prescribed Burns at Calaveras Big Trees State Park (9/24/25), https://www.parks.ca.gov/news/25732

In [137]:
q_info = "Tell me about Calaveras Big Trees."
q_weather = "What is the weather like in Calaveras Big Trees?"

In [138]:
small_agent.run(q_info)

'Calaveras Big Trees State Park is a California state park that is open from sunrise to sunset for day use, and camping is available as of May 15th. The park allows dogs in campgrounds and on fire roads, and offers activities such as camping, fishing, hiking, and swimming. It protects giant sequoias and has a reopened South Grove Trail as of May 1, 2025. The park is not suitable for trailers and large motorhomes due to steep roads and winter closures. For more information, visit <https://www.parks.ca.gov/?page_id=551>.'

In [129]:
large_agent.run(q_info)

'Calaveras Big Trees State Park is a California state park that features giant sequoias, pines, rivers, creeks, wildlife, and meadows, and offers activities such as camping, fishing, hiking, and swimming. The park has a Visitor Center with a museum, film, and gift shop, and dogs are allowed in campgrounds and on fire roads. Currently, the weather at the park is overcast with a temperature of 42°F, humidity of 88%, and wind of 6 mph, with no precipitation.'

## Test both agents on those queries